<a href="https://colab.research.google.com/github/nqthaivl/Copy-Folder-Google-Drive-to-Google-Drive/blob/main/Copy_Folder_Google_Drive_to_Google_Drive.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Copy Folder Google Drive to Google Drive - 1TouchPro

In [ ]:
#@title Input
from ipywidgets import widgets

dest_text = widgets.Text(description="Your drive", placeholder='Nhập đường link folder Google Drive của bạn')
source_text = widgets.Text(description="Shared drive", placeholder='Nhập đường link folder Google Drive shared')
from_page_text = widgets.Text(description="Từ trang", value="0")
to_page_text = widgets.Text(description="Đến trang", value="0")
max_download_size_text = widgets.Text(description="Tổng dung lượng tối đa(GB)", value="700")
exclude_str_text = widgets.Text(description="Bỏ file, folder có chứa nội dung", value="")

display(dest_text)
display(source_text)
display(from_page_text)
display(to_page_text)
display(max_download_size_text)
display(exclude_str_text)

In [ ]:
#@title Run
import time
import re
from googleapiclient.discovery import build
from google.colab import auth


class SizeLimitExceeded(Exception):
    """Raised when the total copied size exceeds the configured limit."""
    pass


def _escape(value):
    """Escape backslashes and single quotes for Drive query strings."""
    return value.replace("\\", "\\\\").replace("'", "\\'")


def _is_transient(error):
    """Return True for errors worth retrying (rate limits / server errors)."""
    status = getattr(getattr(error, "resp", None), "status", None)
    if status in (429, 500, 502, 503, 504):
        return True
    if status == 403:
        msg = str(error).lower()
        return ("rate" in msg) or ("quota" in msg) or ("userratelimit" in msg)
    return False


class DownloadFromDrive:
    def __init__(self):
        self._total_size = 0
        self._limit_size = 0
        self.excluded_strings = []

    def get_user_credential(self):
        auth.authenticate_user()
        drive_service = build('drive', 'v3')
        return drive_service

    def _execute_with_retry(self, request, description, max_retries=5):
        """Execute an API request, retrying transient errors with backoff."""
        delay = 1
        for attempt in range(1, max_retries + 1):
            try:
                return request.execute()
            except Exception as e:
                if attempt >= max_retries or not _is_transient(e):
                    raise
                print(f"Transient error on {description} "
                      f"(attempt {attempt}/{max_retries}): {e}. Retrying in {delay}s...")
                time.sleep(delay)
                delay = min(delay * 2, 32)

    def get_childs_from_folder(self, drive_service, folder_id, from_page, to_page):
        files = []
        page_token = None
        query = f"'{folder_id}' in parents and trashed = false"
        if self.excluded_strings and len(self.excluded_strings) > 0:
            not_contains_query = " and ".join(
                [f"not name contains '{_escape(ext)}'" for ext in self.excluded_strings])
            query += f" and ({not_contains_query})"

        # Normalize the lower bound: pages is 1-indexed, 0 means "from the start".
        lower_page = from_page if from_page > 0 else 1

        pages = 0
        while True:
            try:
                pages += 1
                request = drive_service.files().list(q=query,
                                        orderBy='name, createdTime',
                                        fields='files(id, name, mimeType, size), nextPageToken',
                                        pageToken=page_token,
                                        supportsAllDrives=True,
                                        includeItemsFromAllDrives=True)
                response = self._execute_with_retry(request, "listing folder")

                # Inclusive 1-indexed range; to_page == 0 means "until the end".
                if pages >= lower_page and (to_page == 0 or pages <= to_page):
                    files.extend(response.get('files', []))

                page_token = response.get('nextPageToken', None)
                if page_token is None or (to_page > 0 and pages >= to_page):
                    break
            except Exception as e:
                print(f"An error occurred while listing folder: {str(e)}")
                break

        print(f"Total files: {len(files)}")
        return files

    def copy_file(self, drive_service, dest_folder_id, source_file):
        if source_file['mimeType'] != 'application/vnd.google-apps.folder':
            body_file_inf = {'name': source_file['name'], 'parents': [dest_folder_id]}

            if not self.check_if_exists(drive_service, dest_folder_id, source_file['name']):
                try:
                    start_time = time.time()
                    request = drive_service.files().copy(body=body_file_inf, fileId=source_file['id'],
                                                         supportsAllDrives=True)
                    self._execute_with_retry(request, f"copying {source_file['name']}")
                    end_time = time.time()

                    fileSize = int(source_file.get('size', 0))
                    size_mb = fileSize / (1024 * 1024)
                    self._total_size += size_mb
                    elapsed = end_time - start_time
                    speed_mb = size_mb / elapsed if elapsed > 0 else 0
                    print(f"[{source_file['name']}] copied. Size {size_mb:0.2f} MB. Speed {speed_mb:0.2f} MB/s")

                    if self._total_size >= (self._limit_size * 1024):
                        self.on_total_size_exceeded(f"Total size exceeds {self._limit_size} GB. Ending the program.")
                except SizeLimitExceeded:
                    raise
                except Exception as e:
                    print(f"Failed to copy [{source_file['name']}]: ", e)
            else:
                print(f"[{source_file['name']}] exists.")
        else:
            print(f"Copy at Folder {source_file['name']} Starting")
            try:
                sub_folder_id = self.create_folder(drive_service, dest_folder_id, source_file['name'])
            except RuntimeError as e:
                print(f"Skipping folder '{source_file['name']}': {e}")
                return
            source_files = self.get_childs_from_folder(drive_service, source_file['id'], 0, 0)
            if source_files and len(source_files) > 0:
                self.copy_multiple_files(drive_service, sub_folder_id, source_files)
            print(f"Copy at Folder {source_file['name']} Ending")

    def create_folder(self, drive_service, dest_folder_id, sub_folder_name):
        sub_folder_inf = {'name': sub_folder_name, 'mimeType': 'application/vnd.google-apps.folder', 'parents': [dest_folder_id]}

        exist_folder_id = self.check_if_exists(drive_service, dest_folder_id, sub_folder_name)
        if exist_folder_id:
            return exist_folder_id
        try:
            request = drive_service.files().create(body=sub_folder_inf, fields='id', supportsAllDrives=True)
            folder = self._execute_with_retry(request, f"creating folder {sub_folder_name}")
            return folder['id']
        except Exception as e:
            print("An error occurred: ", e)
            raise RuntimeError(f"Could not create folder '{sub_folder_name}'.")

    def check_if_exists(self, drive_service, dest_folder_id, name):
        try:
            processed_name = _escape(name)

            request = drive_service.files().list(q=f"'{dest_folder_id}' in parents and name = '{processed_name}' and trashed=false",
                                                 fields='files(id)', supportsAllDrives=True, includeItemsFromAllDrives=True)
            results = self._execute_with_retry(request, f"checking existence of {name}")

            if 'files' in results and len(results['files']) > 0:
                return results['files'][0]['id']
        except Exception as e:
            print("An error occurred: ", e)

        return ""

    def copy_multiple_files(self, drive_service, dest_folder_id, source_files):
        for source_file in source_files:
            self.copy_file(drive_service, dest_folder_id, source_file)

    def extract_folder_id_from_url(self, url):
        pattern = r'[-\w]{25,}'
        match = re.search(pattern, url)
        if match:
            return match.group(0)
        else:
            return None

    def on_total_size_exceeded(self, message):
        print(message)
        raise SizeLimitExceeded(message)

    def copy_drive_to_drive(self, destDriveLink, sourceDriveLink, from_page, to_page):
        service = self.get_user_credential()

        start_time = time.time()
        dest_folder_id = self.extract_folder_id_from_url(destDriveLink)
        source_folder_id = self.extract_folder_id_from_url(sourceDriveLink)
        if not dest_folder_id or not source_folder_id:
            print("Invalid Google Drive link. Please check the destination and source links.")
            return

        source_folder = self._execute_with_retry(
            service.files().get(fileId=source_folder_id, supportsAllDrives=True),
            "getting source folder")
        new_dest_folder_id = self.create_folder(service, dest_folder_id, source_folder['name'])

        try:
            source_files = self.get_childs_from_folder(service, source_folder_id, from_page, to_page)
            self.copy_multiple_files(service, new_dest_folder_id, source_files)
        except SizeLimitExceeded:
            pass
        end_time = time.time()

        elapsed = end_time - start_time
        size_gb = self._total_size / 1024
        speed_mb = self._total_size / elapsed if elapsed > 0 else 0

        print(f"Done. Total Size {size_gb:0.2f} GB. Total Time {int(elapsed)} s. SpeedMB {speed_mb:0.2f} MB/s")


# Main
destDriveLink = dest_text.value
sourceDriveLink = source_text.value
fromPage = int(from_page_text.value)
toPage = int(to_page_text.value)

downloader = DownloadFromDrive()
downloader._limit_size = float(max_download_size_text.value)
downloader.excluded_strings = [ext.strip() for ext in exclude_str_text.value.split(",") if ext.strip()]
downloader.copy_drive_to_drive(destDriveLink, sourceDriveLink, fromPage, toPage)
